# 🗞️ Preprocessing Dataset Kompas.com
**Penelitian:** Analisis Sentimen & Topic Modeling Terhadap Diskursus Publik Mengenai Perlindungan Data Pribadi di Indonesia (2020–2025)

**Alur preprocessing:**
Raw Data → Remove Unused Column → Casefolding → Remove Special Characters → Remove Duplicates → Remove Empty Rows → Remove Stopwords → Stemming → Context Filtering → Labeling → Data Splitting


In [ ]:
# CELL 1 — Install Library
!pip install PySastrawi nltk scikit-learn tqdm --quiet

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 210.6/210.6 kB 5.6 MB/s eta 0:00:00


In [ ]:
# CELL 2 — Import Library
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from sklearn.model_selection import train_test_split
from tqdm.auto import tqdm

nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)

print("✅ Library berhasil di-import")

✅ Library berhasil di-import


In [ ]:
# CELL 3 — Load Raw Data Kompas
# Upload file kompas_738.csv terlebih dahulu
# from google.colab import files
# uploaded = files.upload()

df_raw = pd.read_csv('kompas_738.csv')
print(f"✅ Raw data Kompas berhasil dimuat")
print(f"   Shape awal  : {df_raw.shape}")
print(f"   Kolom       : {df_raw.columns.tolist()}")
print(f"\nMissing values per kolom:")
print(df_raw.isnull().sum())
print()
df_raw.head(2)

✅ Raw data Kompas berhasil dimuat
   Shape awal  : (738, 8)
   Kolom       : ['source', 'keyword', 'title', 'date', 'year', 'url', 'content', 'source_file']

Missing values per kolom:
source         0
keyword        0
title          0
date           0
year           0
url            0
content        0
source_file    0
dtype: int64



,source,keyword,title,date,year,url,content,source_file
0,Kompas,keamanan siber Indonesia,Resolusi Penanggulangan Terorisme yang Dipraka...,12/30/2020,2020,https://nasional.kompas.com/read/2020/12/30/10...,"JAKARTA, KOMPAS.com - Resolusi penanggulangan ...",kompas_pdp_2020_109.csv
1,Kompas,keamanan siber Indonesia,"Tren Teknologi 2021, dari Hybrid Cloud, AI, hi...",12/17/2020,2020,https://tekno.kompas.com/read/2020/12/17/19050...,KOMPAS.com - Perusahaan teknologi Internationa...,kompas_pdp_2020_109.csv


In [ ]:
# CELL 4 — Remove Unused Columns
# Kompas sudah lebih bersih (8 kolom), pilih 5 kolom utama

KOLOM_PAKAI = ['date', 'title', 'content', 'year', 'url']

df = df_raw[KOLOM_PAKAI].copy()
print(f"📌 LANGKAH 1 — Remove Unused Columns")
print(f"   Kolom dipertahankan        : {KOLOM_PAKAI}")
print(f"   Shape setelah seleksi kolom: {df.shape}")
df.head(2)

📌 LANGKAH 1 — Remove Unused Columns
   Kolom dipertahankan        : ['date', 'title', 'content', 'year', 'url']
   Shape setelah seleksi kolom: (738, 5)


,date,title,content,year,url
0,12/30/2020,Resolusi Penanggulangan Terorisme yang Dipraka...,"JAKARTA, KOMPAS.com - Resolusi penanggulangan ...",2020,https://nasional.kompas.com/read/2020/12/30/10...
1,12/17/2020,"Tren Teknologi 2021, dari Hybrid Cloud, AI, hi...",KOMPAS.com - Perusahaan teknologi Internationa...,2020,https://tekno.kompas.com/read/2020/12/17/19050...


In [ ]:
# CELL 5 — Casefolding
def casefolding(text):
    """Ubah semua teks menjadi huruf kecil."""
    if pd.isna(text):
        return text
    return str(text).lower()

df['title_clean']   = df['title'].apply(casefolding)
df['content_clean'] = df['content'].apply(casefolding)

print(f"📌 LANGKAH 2 — Casefolding")
print(f"   Sebelum : {df['title'].iloc[0][:70]}")
print(f"   Sesudah : {df['title_clean'].iloc[0][:70]}")

📌 LANGKAH 2 — Casefolding
   Sebelum : Resolusi Penanggulangan Terorisme yang Diprakarsai Indonesia Disahkan 
   Sesudah : resolusi penanggulangan terorisme yang diprakarsai indonesia disahkan 


In [ ]:
# CELL 6 — Remove Special Characters
def remove_special_chars(text):
    """Hapus karakter khusus, HTML, URL, encoding rusak."""
    if pd.isna(text):
        return text
    text = re.sub(r'<[^>]+>', ' ', text)                        # Tag HTML
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)       # URL
    text = re.sub(r'[^\x00-\x7F\u00C0-\u024F]', ' ', text)  # Encoding rusak
    text = re.sub(r'[^a-zA-Z0-9\s]', ' ', text)                # Karakter khusus
    text = re.sub(r'\s+', ' ', text).strip()                    # Whitespace ganda
    return text

df['title_clean']   = df['title_clean'].apply(remove_special_chars)
df['content_clean'] = df['content_clean'].apply(remove_special_chars)

# Hapus noise khas Kompas
noise_pattern = r'(artikel ini telah tayang di|baca juga|lihat juga|dapatkan update|kompasiana adalah platform|simak breaking news)[^\n]*'
df['content_clean'] = df['content_clean'].apply(
    lambda x: re.sub(noise_pattern, ' ', str(x), flags=re.IGNORECASE) if pd.notna(x) else x
)
df['content_clean'] = df['content_clean'].apply(
    lambda x: re.sub(r'\s+', ' ', str(x)).strip() if pd.notna(x) else x
)

print(f"📌 LANGKAH 3 — Remove Special Characters")
print(f"   Contoh (80 char): {df['content_clean'].iloc[0][:80]}")

📌 LANGKAH 3 — Remove Special Characters
   Contoh (80 char): jakarta kompas com resolusi penanggulangan terorisme yang diprakarsai indonesia 


In [ ]:
# CELL 7 — Remove Duplicates
# Cek duplikat berdasarkan title DAN URL
jumlah_sebelum = len(df)

df = df.drop_duplicates(subset=['title_clean'], keep='first')
df = df.drop_duplicates(subset=['url'], keep='first')
df = df.reset_index(drop=True)

jumlah_sesudah = len(df)
print(f"📌 LANGKAH 4 — Remove Duplicates")
print(f"   Sebelum  : {jumlah_sebelum}")
print(f"   Duplikat : {jumlah_sebelum - jumlah_sesudah}")
print(f"   Sesudah  : {jumlah_sesudah}")

📌 LANGKAH 4 — Remove Duplicates
   Sebelum  : 738
   Duplikat : 0
   Sesudah  : 738


In [ ]:
# CELL 8 — Remove Empty Rows
# Kompas: filter juga artikel dengan content terlalu pendek (<50 karakter)
jumlah_sebelum = len(df)

df['content_clean'] = df['content_clean'].replace('', np.nan)
df['title_clean']   = df['title_clean'].replace('', np.nan)
df = df[df['content_clean'].str.len().fillna(0) >= 50]
df = df.dropna(subset=['content_clean', 'title_clean']).reset_index(drop=True)

jumlah_sesudah = len(df)
print(f"📌 LANGKAH 5 — Remove Empty Rows")
print(f"   Sebelum       : {jumlah_sebelum}")
print(f"   Baris dibuang : {jumlah_sebelum - jumlah_sesudah}")
print(f"   Sesudah       : {jumlah_sesudah}")

📌 LANGKAH 5 — Remove Empty Rows
   Sebelum       : 738
   Baris dibuang : 0
   Sesudah       : 738


In [ ]:
# CELL 9 — Remove Stopwords
nltk_sw     = set(stopwords.words('indonesian'))
sastrawi_f  = StopWordRemoverFactory()
sastrawi_sw = set(sastrawi_f.get_stop_words())

custom_sw = {
    'nbsp','amp','quot','com','id','http','www',
    'kompas','kompascom','kompasiana','foto','video',
    'baca','juga','halaman','selanjutnya','sebelumnya',
    'share','komentar','reporter','editor','sumber',
    'tayang','update','redaksi','wartawan','kali','ini'
}
all_stopwords = nltk_sw | sastrawi_sw | custom_sw

def remove_stopwords(text):
    if pd.isna(text): return text
    tokens = text.split()
    return ' '.join([w for w in tokens if w not in all_stopwords and len(w) > 1])

df['title_sw']   = df['title_clean'].apply(remove_stopwords)
df['content_sw'] = df['content_clean'].apply(remove_stopwords)

print(f"📌 LANGKAH 6 — Remove Stopwords")
print(f"   Total stopwords  : {len(all_stopwords)}")
print(f"   Contoh sebelum   : {df['content_clean'].iloc[0][:100]}")
print(f"   Contoh sesudah   : {df['content_sw'].iloc[0][:100]}")

📌 LANGKAH 6 — Remove Stopwords
   Total stopwords  : 838
   Contoh sebelum   : jakarta kompas com resolusi penanggulangan terorisme yang diprakarsai indonesia telah disahkan oleh 
   Contoh sesudah   : jakarta resolusi penanggulangan terorisme diprakarsai indonesia disahkan dewan keamanan perserikatan


In [ ]:
# CELL 10 — Stemming (Sastrawi)
stemmer_factory = StemmerFactory()
stemmer = stemmer_factory.create_stemmer()

def stemming(text):
    if pd.isna(text): return text
    return stemmer.stem(str(text))

tqdm.pandas(desc="Stemming Kompas")

print("📌 LANGKAH 7 — Stemming (Sastrawi)")
print("   ⚠️ Harap tunggu beberapa menit...")

df['title_stem']   = df['title_sw'].progress_apply(stemming)
df['content_stem'] = df['content_sw'].progress_apply(stemming)

print("   ✅ Stemming selesai!")
print(f"   Sebelum: {df['content_sw'].iloc[0][:80]}")
print(f"   Sesudah: {df['content_stem'].iloc[0][:80]}")

📌 LANGKAH 7 — Stemming (Sastrawi)
   ⚠️ Harap tunggu beberapa menit...


Stemming Kompas:   0%|          | 0/738 [00:00<?, ?it/s]

Stemming Kompas:   0%|          | 0/738 [00:00<?, ?it/s]

   ✅ Stemming selesai!
   Sebelum: jakarta resolusi penanggulangan terorisme diprakarsai indonesia disahkan dewan k
   Sesudah: jakarta resolusi tanggulang terorisme prakarsa indonesia sah dewan aman serikat 


In [ ]:
# CELL 11 — Context Filtering
# kata_kunci_relevan = [
#     'data pribadi', 'pdp', 'ruu pdp', 'uu pdp',
#     'perlindungan data', 'kebocoran data', 'privasi',
#     'privacy', 'personal data', 'data protection',
#     'penyalahgunaan data', 'keamanan data', 'data sensitif',
#     'kominfo', 'bssn', 'siber', 'cyber', 'hacker', 'phishing',
#     'identitas digital', 'consent', 'gdpr', 'regulasi data'
# ]
kata_kunci_relevan = [
    'data pribadi',
    'perlindungan data pribadi',
    'uu pdp',
    'privasi',
    'keamanan data',
    'data'
]

def is_relevan(row):
    teks = str(row['title']).lower() + ' ' + str(row['content']).lower()
    return any(kw in teks for kw in kata_kunci_relevan)

jumlah_sebelum = len(df)
df['relevan']   = df.apply(is_relevan, axis=1)
df_filtered     = df[df['relevan']].copy().reset_index(drop=True)
jumlah_sesudah  = len(df_filtered)

print(f"📌 LANGKAH 8 — Context Filtering")
print(f"   Sebelum filter   : {jumlah_sebelum}")
print(f"   Tidak relevan    : {jumlah_sebelum - jumlah_sesudah}")
print(f"   Lolos filter     : {jumlah_sesudah}")

📌 LANGKAH 8 — Context Filtering
   Sebelum filter   : 738
   Tidak relevan    : 175
   Lolos filter     : 563


In [ ]:
df.to_csv("kompas_ready_label_563.csv", index=False)

print("kompas_ready_label berhasil disimpan")

kompas_ready_label berhasil disimpan


In [ ]:
df.head(3)

,date,title,content,year,url,title_clean,content_clean,title_sw,content_sw,title_stem,content_stem,relevan
0,12/30/2020,Resolusi Penanggulangan Terorisme yang Dipraka...,"JAKARTA, KOMPAS.com - Resolusi penanggulangan ...",2020,https://nasional.kompas.com/read/2020/12/30/10...,resolusi penanggulangan terorisme yang dipraka...,jakarta kompas com resolusi penanggulangan ter...,resolusi penanggulangan terorisme diprakarsai ...,jakarta resolusi penanggulangan terorisme dipr...,resolusi tanggulang terorisme prakarsa indones...,jakarta resolusi tanggulang terorisme prakarsa...,False
1,12/17/2020,"Tren Teknologi 2021, dari Hybrid Cloud, AI, hi...",KOMPAS.com - Perusahaan teknologi Internationa...,2020,https://tekno.kompas.com/read/2020/12/17/19050...,tren teknologi 2021 dari hybrid cloud ai hingg...,kompas com perusahaan teknologi international ...,tren teknologi 2021 hybrid cloud ai keamanan s...,perusahaan teknologi international business ma...,tren teknologi 2021 hybrid cloud ai aman siber,usaha teknologi international business machine...,True
2,12/11/2020,5 Cara Menjaga Data Pribadi Tetap Aman,"JAKARTA, KOMPAS.com - Kasus kebocoran data pri...",2020,https://money.kompas.com/read/2020/12/11/17512...,5 cara menjaga data pribadi tetap aman,jakarta kompas com kasus kebocoran data pribad...,menjaga data pribadi aman,jakarta kebocoran data pribadi platform teknol...,jaga data pribadi aman,jakarta bocor data pribadi platform teknologi ...,True
